# 暗槓判断の頻度と教材の優先度

## tl;dr
通常の牌効率問題を優先する。暗槓は任意のテーマ練習候補。合法機会はツモ後判断の約0.67%、現モードに近い局面の約0.94%。

## Context & Methods
判断前の合法候補を数える。実行率は最適率ではない。2018–19〜2025–26の保存牌譜のうち、南4以降などを除いた20,446局。

### Key Assumptions
局家牌種の重複除去は同じ四枚の反復保有をまとめる。構造条件一致は全出題品質条件の合格ではない。元データ再構築はanalyze.pyで行う。

## Data
3本のgzipのhashと原牌譜8本を主集計で照合済み。ここではfull/summary.jsonとgenerated-sample.json、verification.jsonを読み直す。

In [1]:
import json
from pathlib import Path
base = Path.cwd()
if not (base / 'full/summary.json').exists():
    base = base / 'calibration/ankan-opportunity-analysis'
summary = json.loads((base / 'full/summary.json').read_text(encoding='utf-8'))
generated = json.loads((base / 'generated-sample.json').read_text(encoding='utf-8'))
validation = json.loads((base / 'verification.json').read_text(encoding='utf-8'))
assert validation['status'] == 'pass'
print('教師窓:', summary['totals']['teacher_windows'])
print('局:', summary['totals']['rounds'])
print('保存生成問題:', len(generated['records']))

教師窓: 2085155
局: 20446
保存生成問題: 200


## Results
### 判断機会の割合
分母が異なるため率を直接足さない。小さな比較なので、図の代わりに正確な分子と分母の表を使う。

In [2]:
t = summary['totals']
comparisons = [
 ('ツモ後', t['ankan_windows'], t['self_action_after_live']+t['self_action_after_rinshan']),
 ('局家', t['unique_ankan_seat_rounds'], t['seat_rounds']),
 ('2→1の構造条件', t['app_structural_ankan'], t['app_structural_windows']),
 ('字牌候補除外後', t['app_no_honor_candidate_ankan'], t['app_no_honor_candidate_windows'])]
for label,n,d in comparisons:
    print(f'{label}: {n:,} / {d:,} = {n/d*100:.3f}%')
assert t['ankan_windows'] == 6868 and t['observed_ankan'] == 1015

ツモ後: 6,868 / 1,021,676 = 0.672%
局家: 2,241 / 81,784 = 2.740%
2→1の構造条件: 618 / 65,962 = 0.937%
字牌候補除外後: 402 / 38,696 = 1.039%


### 出題器の200問標本
人工的な出題分布であり、実戦頻度の推定ではない。

In [3]:
for key in ['honorDiscardCandidate','correctHonorDiscard','anyQuad','quadOrHonorCandidate']:
    count = sum(r['flags'][key] for r in generated['records'])
    assert count == generated['aggregate']['counts'][key]
    print(f'{key}: {count}/200 = {count/2:.1f}%')
print('JavaScriptとの独立照合標本:', validation['checkedDistinctHands'])

honorDiscardCandidate: 70/200 = 35.0%
correctHonorDiscard: 67/200 = 33.5%
anyQuad: 12/200 = 6.0%
quadOrHonorCandidate: 77/200 = 38.5%
JavaScriptとの独立照合標本: 198


## Takeaways
字牌候補を含む問題の整理を先に行い、暗槓を頻繁に通常枠へ混ぜる開発は後回しにする。代表例のテーマ練習は残す。誤答率や打点込み損失は未測定であり、最適な教材比率や個々の暗槓の正解を示す調査ではない。詳細と再現手順はFINDINGS.md。